# Libs and config

In [12]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score,
    recall_score,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    confusion_matrix_at_thresholds,
    roc_auc_score,
)

from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split, TimeSeriesSplit
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

import os

os.chdir("..")

# Data

In [2]:
df = pd.read_csv("data/df_cleaned.csv")

display(df.head())

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,...,Not Specified,Debt Consolidation Loan,Payday Loan,Auto Loan,Personal Loan,Credit-Builder Loan,Student Loan,Mortgage Loan,Home Equity Loan,Credit_Score_binomial
0,CUS_0x1000,202201,17,30625.94,2706.161667,6,27,2,62,1.63,...,0,0,0,0,0,1,0,0,1,0
1,CUS_0x1000,202202,17,30625.94,2706.161667,6,27,2,62,1.63,...,0,0,0,0,0,1,0,0,1,1
2,CUS_0x1000,202203,17,30625.94,2706.161667,6,27,2,62,1.63,...,0,0,0,0,0,1,0,0,1,1
3,CUS_0x1000,202204,17,30625.94,2706.161667,6,27,2,64,1.63,...,0,0,0,0,0,1,0,0,1,1
4,CUS_0x1000,202205,17,30625.94,2706.161667,6,27,2,67,2.63,...,0,0,0,0,0,1,0,0,1,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 32 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Customer_ID               100000 non-null  str    
 1   num_month                 100000 non-null  int64  
 2   Age                       100000 non-null  int64  
 3   Annual_Income             100000 non-null  float64
 4   Monthly_Inhand_Salary     100000 non-null  float64
 5   Num_Bank_Accounts         100000 non-null  int64  
 6   Interest_Rate             100000 non-null  int64  
 7   Num_of_Loan               100000 non-null  int64  
 8   Delay_from_due_date       100000 non-null  int64  
 9   Changed_Credit_Limit      100000 non-null  float64
 10  Num_Credit_Inquiries      100000 non-null  float64
 11  Outstanding_Debt          100000 non-null  float64
 12  Credit_Utilization_Ratio  100000 non-null  float64
 13  Credit_History_Age        100000 non-null  float64
 14  

In [14]:
df = df.drop(columns=["Customer_ID"])

In [15]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.to_list()
categorical_cols

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_12571/182059991.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.to_list()


['Occupation', 'Credit_Mix', 'Payment_Behaviour', 'Payment_of_Min_Amount']

In [18]:
encoder = TargetEncoder(smooth="auto", random_state=42)

# Fast modelling

## split

In [26]:
split_month = 202206
target_col = "Credit_Score_binomial"
train = df[df["num_month"] <= split_month].drop(columns=["num_month"])
test = df[df["num_month"] > split_month].drop(columns=["num_month"])

# TODO: use masks instead
X_train = train.drop(columns=[target_col])
y_train = train[target_col]

X_test = test.drop(columns=[target_col])
y_test = test[target_col]

X_train_encoded = X_train.copy()
X_train_encoded[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols], y_train
)
X_test_encoded = X_test.copy()
X_test_encoded[categorical_cols] = encoder.transform(X_test[categorical_cols])

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(75000, 29)
(75000,)
(25000, 29)
(25000,)


In [27]:
# Print the unique categories the encoder discovered
print("Categories learned:", encoder.categories_)

# Print the actual encoded numerical values assigned to those categories
print("Encoded numeric mappings:", encoder.encodings_)

Categories learned: [array(['Accountant', 'Architect', 'Developer', 'Doctor', 'Engineer',
       'Entrepreneur', 'Journalist', 'Lawyer', 'Manager', 'Mechanic',
       'Media_Manager', 'Musician', 'Scientist', 'Teacher', 'Writer'],
      dtype=object), array(['Bad', 'Good', 'Standard'], dtype=object), array(['High_spent_Large_value_payments',
       'High_spent_Medium_value_payments',
       'High_spent_Small_value_payments',
       'Low_spent_Large_value_payments',
       'Low_spent_Medium_value_payments',
       'Low_spent_Small_value_payments'], dtype=object), array(['No', 'Yes'], dtype=object)]
Encoded numeric mappings: [array([0.30011667, 0.27511036, 0.2873022 , 0.27081177, 0.3088541 ,
       0.30725684, 0.2796839 , 0.28091119, 0.2862778 , 0.30224083,
       0.26230683, 0.27938933, 0.31197658, 0.30415386, 0.29949043]), array([0.61559669, 0.16026352, 0.20810987]), array([0.21977387, 0.25973865, 0.27824771, 0.28409583, 0.3002082 ,
       0.35357768]), array([0.131075  , 0.39919849])]

## helper functions

In [51]:
def evaluate_classification_model(
    y_true,
    y_pred,
    y_proba,
    model_name: str,
) -> dict:

    y_true_ser = pd.Series(y_true).reset_index(drop=True)
    y_proba_ser = pd.Series(y_proba).reset_index(drop=True)

    proba_goods = y_proba_ser[y_proba_ser == 0]
    proba_bads = y_proba_ser[y_proba_ser == 1]

    ks_stat = ks_2samp(proba_goods, proba_bads)
    report = {
        "model": model_name,
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "PR_AUC": average_precision_score(y_true, y_proba),
        "Accuracy": accuracy_score(y_true, y_pred),
        "KS": ks_stat,
    }

    return pd.DataFrame([report])

In [52]:
model.__class__.__name__

'LogisticRegression'

In [54]:
xgb = XGBClassifier(random_state=42, n_jobs=1)
lgbm = LGBMClassifier(random_state=42, n_jobs=1, verbose=-1)
rdmf = RandomForestClassifier(random_state=42, n_jobs=1)
lgr = LogisticRegression(
    random_state=42,
)
models = [xgb, lgbm, rdmf, lgr]

results = []

for model in models:

    model.fit(X_train_encoded, y_train)

    y_pred = model.predict(X_test_encoded)
    y_proba = model.predict_proba(X_test_encoded)[:, 1]

    results.append(
        evaluate_classification_model(y_test, y_pred, y_proba, model.__class__.__name__)
    )

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_12571/3205770137.py:14: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  ks_stat = ks_2samp(proba_goods, proba_bads)
/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_12571/3205770137.py:14: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  ks_stat = ks_2samp(proba_goods, proba_bads)
/Users/luisfernandocorcueraleon/Desktop/code/Credit-Score-Classification/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocess

In [55]:
report_df = pd.concat(results, ignore_index=True)
report_df

,model,Precision,Recall,ROC_AUC,PR_AUC,Accuracy,KS
0,XGBClassifier,0.698485,0.683481,0.861356,0.687798,0.82348,"(nan, nan)"
1,LGBMClassifier,0.685916,0.618902,0.835303,0.660806,0.80820,"(nan, nan)"
2,RandomForestClassifier,0.719069,0.732123,0.880445,0.717877,0.84012,"(1.0, 0.001974333662388942)"
3,LogisticRegression,0.522537,0.319706,0.683947,0.444596,0.71932,"(nan, nan)"
